# Drawing interpolation extension

Port of `apps/demo-ext-drawing/drawing.html`. Demonstrates the `@niivue/nv-ext-drawing` interpolation workflow: draw on a few non-adjacent slices, then fill the gaps automatically.

**Workflow:**
1. Click **Enable drawing** and choose a pen color/size.
2. Draw on at least two slices that are several slices apart (along the chosen axis).
3. Click **Find boundaries** to confirm the first/last drawn slice was detected.
4. Click **Interpolate** to fill the gaps. Toggle **Intensity-guided** to use the background volume's intensity to guide the interpolation along anatomical edges.

Heavy work runs in a Web Worker on the JS side; Python awaits a small summary dict (`{before, after, elapsed_ms}` for interpolate, `{first, last, elapsed_ms}` for find-boundaries).

The hover-driven magic-wand demo (`apps/demo-ext-drawing/magic-wand.ts`) is not ported — its sub-frame pointer-preview UX doesn't fit the kernel round-trip model. A programmatic `magic_wand(seed, ...)` API may follow.


In [ ]:
import asyncio

import ipywidgets as widgets
from IPython.display import display

from ipyniivue import NiiVue

BASE_URL = "https://niivue.github.io/mono"
VOLUMES = f"{BASE_URL}/volumes"

nv = NiiVue(
    background_color=[0.1, 0.1, 0.1, 1.0],
    is_colorbar_visible=False,
    slice_type=3,
    backend="webgl2",
)

# --- Drawing controls ---
enable_btn = widgets.ToggleButton(value=False, description="Enable drawing", icon="pencil")
pen_color = widgets.Dropdown(
    options=[("Red (1)", 1), ("Green (2)", 2), ("Blue (3)", 3), ("Yellow (4)", 4), ("Cyan (5)", 5)],
    value=1,
    description="Color",
)
pen_size = widgets.IntSlider(value=3, min=1, max=10, description="Size")
undo_btn = widgets.Button(description="Undo", icon="undo")

# --- Interpolation controls ---
axis_select = widgets.Dropdown(
    options=[("Axial", 0), ("Coronal", 1), ("Sagittal", 2)],
    value=0,
    description="Axis",
)
intensity_guided = widgets.Checkbox(value=False, description="Intensity-guided")
alpha_slider = widgets.FloatSlider(
    value=0.70, min=0.0, max=1.0, step=0.01, description="\u03b1 weight"
)
sigma_slider = widgets.FloatSlider(
    value=0.10, min=0.01, max=1.0, step=0.01, description="\u03c3 sigma"
)
thresh_slider = widgets.FloatSlider(
    value=0.38, min=0.01, max=1.0, step=0.01, description="threshold"
)
smooth = widgets.Checkbox(value=True, description="Smooth slices")
find_btn = widgets.Button(description="Find boundaries")
interp_btn = widgets.Button(description="Interpolate", button_style="primary")

status = widgets.Label(value="Loading volume\u2026")

intensity_box = widgets.HBox(
    [alpha_slider, sigma_slider, thresh_slider, smooth],
    layout=widgets.Layout(display="none"),
)


def on_intensity_toggle(change):
    intensity_box.layout.display = "flex" if change["new"] else "none"


def on_enable(change):
    if change["new"]:
        nv.create_empty_drawing()
        nv.draw_is_enabled = True
        enable_btn.description = "Disable drawing"
        status.value = "Drawing enabled. Draw on a few slices, then Interpolate."
    else:
        nv.draw_is_enabled = False
        enable_btn.description = "Enable drawing"
        status.value = "Drawing disabled."


def on_pen_color(change):
    nv.draw_pen_value = change["new"]


def on_pen_size(change):
    nv.draw_pen_size = change["new"]


def on_undo(_btn):
    nv.draw_undo()
    status.value = "Undo."


async def do_find():
    axis = axis_select.value
    axis_name = ["Axial", "Coronal", "Sagittal"][axis]
    status.value = f"Finding {axis_name} boundaries\u2026"
    try:
        result = await nv.find_drawing_boundary_slices(axis)
    except Exception as exc:
        status.value = f"Find boundaries failed: {exc}"
        return
    if not result:
        status.value = f"No drawing data along {axis_name} axis. Draw on at least one slice first."
        return
    status.value = (
        f"{axis_name} boundaries: first={result['first']}, last={result['last']} "
        f"({result['elapsed_ms']:.0f} ms)"
    )


def on_find(_btn):
    asyncio.ensure_future(do_find())


async def do_interp():
    axis = axis_select.value
    axis_name = ["Axial", "Coronal", "Sagittal"][axis]
    interp_btn.disabled = True
    status.value = f"Interpolating {axis_name} slices\u2026"
    try:
        result = await nv.interpolate_drawing_slices(
            axis=axis,
            use_intensity_guided=intensity_guided.value,
            intensity_weight=alpha_slider.value,
            intensity_sigma=sigma_slider.value,
            binary_threshold=thresh_slider.value,
            apply_smoothing_to_slices=smooth.value,
        )
        status.value = (
            f"{axis_name} interpolation: {result['before']} \u2192 {result['after']} voxels "
            f"({result['elapsed_ms']:.0f} ms, worker)"
        )
    except Exception as exc:
        status.value = f"Interpolate failed: {exc}"
    finally:
        interp_btn.disabled = False


def on_interp(_btn):
    asyncio.ensure_future(do_interp())


enable_btn.observe(on_enable, names="value")
pen_color.observe(on_pen_color, names="value")
pen_size.observe(on_pen_size, names="value")
intensity_guided.observe(on_intensity_toggle, names="value")
undo_btn.on_click(on_undo)
find_btn.on_click(on_find)
interp_btn.on_click(on_interp)

controls = widgets.VBox(
    [
        widgets.HBox([enable_btn, pen_color, pen_size, undo_btn]),
        widgets.HBox([axis_select, intensity_guided, find_btn, interp_btn]),
        intensity_box,
    ]
)
display(controls)
display(nv)
display(status)

nv.add_volume_from_url(f"{VOLUMES}/mni152.nii.gz")
status.value = "Ready. Click Enable drawing to start."